# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MA-1305/Project_Mahin-1305/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Research paper findings reviewed successfully.")

Research paper findings reviewed successfully.


## Finding 1

The research paper reports that refreshed content is associated with improved search performance.

### Methodology Question

How was the refreshed-content label defined, and was the same definition applied consistently across all pages?

## Finding 2

The paper reports that search-performance signals are associated with higher visibility.

### Methodology Question

Was the evaluation design grouped or time-aware so that information from related pages or future observations could not leak into the evaluation?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# --------------------------------------------------
# Load dataset
# --------------------------------------------------

repo_path = Path("/content/Project_Mahin-1305")

if not repo_path.exists():
    !git clone https://github.com/MA-1305/Project_Mahin-1305.git /content/Project_Mahin-1305

data_path = repo_path / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

# Proxy target from Week 3
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Dataset shape:", df.shape)
print("Declining label rate:",
      round(df["is_declining_label"].mean(), 3))

# --------------------------------------------------
# Features
# --------------------------------------------------

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier"
]

# Keep only columns that actually exist
numeric_features = [
    c for c in numeric_features if c in df.columns
]

categorical_features = [
    c for c in categorical_features if c in df.columns
]

feature_columns = numeric_features + categorical_features

X = df[feature_columns].copy()
y = df["is_declining_label"].copy()

# --------------------------------------------------
# Preprocessing
# --------------------------------------------------

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

def make_model():
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

# --------------------------------------------------
# Helper: Precision@50
# --------------------------------------------------

def precision_at_k(y_true, probabilities, k=50):
    k = min(k, len(y_true))

    top_indices = np.argsort(probabilities)[::-1][:k]

    return y_true.iloc[top_indices].mean()

# --------------------------------------------------
# BEFORE: random split
# --------------------------------------------------

X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

random_model = make_model()

random_model.fit(
    X_train_random,
    y_train_random
)

random_prob = random_model.predict_proba(
    X_test_random
)[:, 1]

random_p50 = precision_at_k(
    y_test_random.reset_index(drop=True),
    random_prob,
    50
)

random_auc = roc_auc_score(
    y_test_random,
    random_prob
)

random_ap = average_precision_score(
    y_test_random,
    random_prob
)

# --------------------------------------------------
# AFTER: client-holdout split
# --------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=df["client_id"]
    )
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

group_model = make_model()

group_model.fit(
    X_train_group,
    y_train_group
)

group_prob = group_model.predict_proba(
    X_test_group
)[:, 1]

group_p50 = precision_at_k(
    y_test_group.reset_index(drop=True),
    group_prob,
    50
)

group_auc = roc_auc_score(
    y_test_group,
    group_prob
)

group_ap = average_precision_score(
    y_test_group,
    group_prob
)

# --------------------------------------------------
# Comparison
# --------------------------------------------------

comparison = pd.DataFrame({
    "Validation": [
        "Random split",
        "Client holdout"
    ],
    "Precision@50": [
        random_p50,
        group_p50
    ],
    "ROC_AUC": [
        random_auc,
        group_auc
    ],
    "Average_Precision": [
        random_ap,
        group_ap
    ]
})

print("\nValidation comparison:")
print(comparison.round(3))

print(
    "\nRandom split test rows:",
    len(y_test_random)
)

print(
    "Client-holdout test rows:",
    len(y_test_group)
)

print(
    "Training clients:",
    df.iloc[train_idx]["client_id"].nunique()
)

print(
    "Testing clients:",
    df.iloc[test_idx]["client_id"].nunique()
)

Dataset shape: (30000, 45)
Declining label rate: 0.542

Validation comparison:
       Validation  Precision@50  ROC_AUC  Average_Precision
0    Random split          0.96    0.757              0.774
1  Client holdout          0.48    0.587              0.577

Random split test rows: 6000
Client-holdout test rows: 6163
Training clients: 25
Testing clients: 7


## Honest Validation

The Week-5 model is compared under a standard random split and a client-holdout split.

The client-holdout split is more conservative because pages from the same client are kept in only one side of the evaluation. This gives a better indication of how the model may perform on unseen clients.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --------------------------------------------------
# Leakage audit
# --------------------------------------------------

print("Leakage Audit")
print("-" * 40)

# Columns that should never be model features
forbidden_columns = [
    "is_declining_label",
    "trend_direction",
    "client_id"
]

print("Forbidden columns checked:")

for col in forbidden_columns:
    print(
        f"- {col}:",
        "PRESENT" if col in feature_columns else "NOT USED"
    )

# Check that forbidden columns are absent
assert not any(
    col in feature_columns
    for col in forbidden_columns
), "Potential target/identifier leakage detected."

# Position-related fields were flagged in W03
position_risk_columns = [
    "avg_position",
    "position_tier"
]

print("\nPosition-related fields:")
for col in position_risk_columns:
    print(
        f"- {col}:",
        "EXCLUDED" if col not in feature_columns else "USED"
    )

# Check client overlap
train_clients = set(
    df.iloc[train_idx]["client_id"]
)

test_clients = set(
    df.iloc[test_idx]["client_id"]
)

overlap = train_clients.intersection(
    test_clients
)

print("\nClient overlap between train and test:", len(overlap))

assert len(overlap) == 0, (
    "Client leakage detected: some clients occur in both splits."
)

print("\nLeakage audit completed.")
print("No target column is used as a feature.")
print("Client groups do not overlap.")
print("Position-related fields are excluded from the final feature set.")

Leakage Audit
----------------------------------------
Forbidden columns checked:
- is_declining_label: NOT USED
- trend_direction: NOT USED
- client_id: NOT USED

Position-related fields:
- avg_position: EXCLUDED
- position_tier: EXCLUDED

Client overlap between train and test: 0

Leakage audit completed.
No target column is used as a feature.
Client groups do not overlap.
Position-related fields are excluded from the final feature set.


## Leakage Audit

I reviewed the final feature set for target-derived columns, future information, and identifiers.

The proxy target `is_declining_label` and the source field `trend_direction` are excluded from the feature matrix.

Client identifiers are used only for grouped validation and are not used as model features.

Position-related fields are treated cautiously because they may encode information closely related to search outcomes. The final feature set therefore excludes `avg_position` and `position_tier`.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Claim rewritten using safe research language.")

Claim rewritten using safe research language.


## Claim Rewrite

### Original Claim

The Random Forest model accurately predicts which pages need content refreshes.

### Rewritten Claim

The Random Forest model measured patterns associated with the declining proxy label in the available dataset. Performance was evaluated using a client-holdout split and Precision@50, so the results should be interpreted as directional decision support rather than proof that refreshing a page will cause better search performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.